# Autodata 3.45 — verificare instrucțiuni originale

Acest notebook citește **doar documentația** de instalare din pachetul 16, în mod inert, fără a rula aplicații, scripturi, keygen-uri sau emulatoare.

Folosește numai descărcarea standard Google Drive. Dacă Google blochează pachetul ca malware/spam, salvează un raport cu **BLOCKED_BY_GOOGLE_DRIVE** și nu încearcă să ocolească protecția.

Rezultatele ajung în folderul separat **Autodata_OneClick_Audit** din Google Drive. Nu trimite coduri de activare sau informații private în GitHub.

Pe telefon: **Runtime → Run all**.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth, io, zipfile, hashlib, json, re, datetime
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaInMemoryUpload
from googleapiclient.errors import HttpError

FOLDER_ID="1XqpDaYI5zkdUXY85wLSvpsklatT0P0Bd"
PART_NAME="autodata_part_0016.zip"
OUT_NAME="Autodata_OneClick_Audit"
creds,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
service=build("drive","v3",credentials=creds,cache_discovery=False)
def list_folder(folder):
    items=[];token=None
    while True:
        r=service.files().list(q="'"+folder+"' in parents and trashed=false",pageSize=100,
                               fields="nextPageToken,files(id,name,size,mimeType)",
                               pageToken=token).execute(num_retries=4)
        items+=r.get("files",[]);token=r.get("nextPageToken")
        if not token:return items
src={f["name"]:f for f in list_folder(FOLDER_ID)}
if PART_NAME not in src:raise RuntimeError("Lipsește pachetul 16 din folder")
found=[f for f in list_folder(FOLDER_ID) if f["name"]==OUT_NAME and
       f.get("mimeType")=="application/vnd.google-apps.folder"]
dest=found[0]["id"] if found else service.files().create(body={"name":OUT_NAME,
         "mimeType":"application/vnd.google-apps.folder","parents":[FOLDER_ID]},
         fields="id").execute(num_retries=4)["id"]
def save(name,data,mime="application/json"):
    if isinstance(data,dict):data=json.dumps(data,ensure_ascii=False,indent=2).encode("utf-8")
    elif isinstance(data,str):data=data.encode("utf-8")
    media=MediaInMemoryUpload(data,mimetype=mime)
    existing=next((f for f in list_folder(dest) if f["name"]==name),None)
    if existing:service.files().update(fileId=existing["id"],media_body=media).execute(num_retries=4)
    else:service.files().create(body={"name":name,"parents":[dest]},media_body=media).execute(num_retries=4)
    print("Saved",name)
print("Output:", "https://drive.google.com/drive/folders/"+dest)

In [ ]:
# Aici se oprește dacă furnizorul blochează descărcarea. Nu folosim acknowledgeAbuse.
result={"archive_part":PART_NAME,"mode":"text_document_inspection_only",
        "executed_original_programs":False,"malware_bypass":False,"status":"pending"}
payload=io.BytesIO()
try:
    req=service.files().get_media(fileId=src[PART_NAME]["id"])
    dl=MediaIoBaseDownload(payload,req,chunksize=4*1024*1024)
    done=False
    while not done:
        progress,done=dl.next_chunk(num_retries=4)
    result["download_bytes"]=payload.tell()
    result["status"]="downloaded_for_inert_review"
except HttpError as exc:
    raw=exc.content.decode("utf-8","replace") if isinstance(exc.content,bytes) else str(exc.content)
    result["status"]="BLOCKED_BY_GOOGLE_DRIVE" if (exc.resp.status==403 and
       any(x in raw for x in ("cannotDownloadAbusiveFile","malware","spam"))) else "API_ERROR"
    result["http_status"]=int(exc.resp.status)
    result["reason"]=raw[:1000]
    save("SOURCE_ACCESS_STATUS.json",result)
    print("SOURCE_ACCESS_STATUS:",result["status"])
except Exception as exc:
    result["status"]="TRANSPORT_ERROR"
    result["reason"]=type(exc).__name__+": "+str(exc)[:1000]
    save("SOURCE_ACCESS_STATUS.json",result)
    print("SOURCE_ACCESS_STATUS:",result["status"])
if result["status"]!="downloaded_for_inert_review":
    print("Arhiva blocată rămâne neatinsă. Nu se extrag sau execută fișiere.")
else:
    save("SOURCE_ACCESS_STATUS.json",result)

In [ ]:
# Se rulează numai dacă Google a autorizat descărcarea normală.
report={"status":"NOT_ANALYZED","documents":[],"excluded":[],"warnings":[]}
if result["status"]=="downloaded_for_inert_review":
    import xml.etree.ElementTree as ET
    import subprocess, tempfile, pathlib
    try:
        from pypdf import PdfReader
    except ImportError:
        import sys
        subprocess.run([sys.executable,"-m","pip","-q","install","pypdf"],check=True)
        from pypdf import PdfReader
    payload.seek(0)
    with zipfile.ZipFile(payload) as z:
        for info in z.infolist():
            name=info.filename
            if info.is_dir():continue
            lower=name.lower()
            suffix=pathlib.PurePosixPath(name).suffix.lower()
            if suffix not in (".txt",".bat",".cmd",".reg",".pdf"):
                report["excluded"].append(name)
                continue
            if info.file_size>2*1024*1024:
                report["warnings"].append("Over 2 MB document not read: "+name)
                continue
            raw=z.read(info)
            item={"name":name,"length":len(raw),"sha256":hashlib.sha256(raw).hexdigest()}
            if suffix==".pdf":
                try:
                    pdf=PdfReader(io.BytesIO(raw),strict=False)
                    text="\n".join((page.extract_text() or "") for page in pdf.pages[:35])
                    item["pages"]=len(pdf.pages)
                except Exception as exc:
                    item["parse_error"]=type(exc).__name__+": "+str(exc)[:160]
                    text=""
            else:
                for encoding in ("utf-8-sig","utf-16","cp1252"):
                    try:text=raw.decode(encoding);break
                    except UnicodeDecodeError:continue
                else:text=raw.decode("latin1","replace")
            # Audit document text; no script runs, no shell commands are evaluated.
            # Avoid copying potential license keys from the private source to the summary.
            def mask(s):
                s=re.sub(r"(?i)(serial|license key|activation key|password)\s*[:=]\s*\S+",
                         r"\1: [REDACTED]",s)
                return s
            item["text"]=mask(text[:60000])
            report["documents"].append(item)
    report["status"]="DOCUMENTS_READ_INERTLY"
    report["document_count"]=len(report["documents"])
    report["license_validity"]="NOT_VERIFIED"
    save("INSTRUCTIONS_AUDIT.json",report)
    # Human-readable text, separated by document.
    txt="\n\n".join("=== "+i["name"]+" ===\n"+i.get("text",i.get("parse_error",""))
                    for i in report["documents"])
    save("INSTRUCTIONS_READ_ONLY.txt",txt,"text/plain")
    print("INSTRUCTIONS_AUDIT=PASS",report["document_count"])
else:
    print("Documentele nu pot fi citite din cauza blocajului: ", result["status"])
print("Folder audit:", "https://drive.google.com/drive/folders/"+dest)